# PERSUADE v8: Enhanced Span Ablation Analysis

**Enhancements over v7:**
1. **Boundary-adjacent spans**: pre_target (80-90%), immediate_pre_target (90-100%)
2. **Larger ablation spans**: 128 tokens (and optionally 256)
3. **Stratified scoring regions**: score_early (512-576), score_mid (576-640), score_late (640-768)
4. **Multiple random ablations**: 10 random spans per essay, averaged
5. **Specific word metrics**: flip_rate + top-k overlap as primary outcomes

**Hypothesis:** Boundary-adjacent ablations will hit score_early hardest, then decay with distance.

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate bitsandbytes pandas numpy matplotlib seaborn tqdm statsmodels

In [ ]:
import json
import math
import os
import time
import random
from datetime import datetime
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from scipy import stats
import statsmodels.formula.api as smf

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Paths
DRIVE_BASE = "/content/drive/MyDrive/LRTIA/Data/persuade_clean"
OUTPUT_BASE = "/content/drive/MyDrive/LRTIA/Results/Persuade"
COHORT_PATH = f"{DRIVE_BASE}/cohorts/persuade_score_long_cohort.jsonl"

EXPERIMENT = 'score_long_span_ablation_v2'

# Model
MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = True  # True for T4, False for A100

# Context/scoring parameters
CONTEXT_LENGTH = 512  # EXTENDED regime burn-in
TOTAL_SCORE_TOKENS = 256  # tokens 512-768

# Stratified scoring regions
SCORE_REGIONS = {
    'score_early': (512, 576),   # first 64 tokens after context
    'score_mid': (576, 640),     # next 64 tokens
    'score_late': (640, 768),    # final 128 tokens
}

# Span locations (as fraction of 512-token context)
# Now includes boundary-adjacent spans
SPAN_CONFIGS = {
    'early': (0.00, 0.10),              # tokens 0-51
    'early_mid': (0.10, 0.20),          # tokens 51-102
    'middle': (0.40, 0.50),             # tokens 205-256
    'late': (0.70, 0.80),               # tokens 358-409
    'pre_target': (0.80, 0.90),         # tokens 409-460
    'immediate_pre_target': (0.90, 1.00),  # tokens 460-512
}

# Ablation span sizes to test
SPAN_SIZES = [128]  # Primary: 128 tokens
# SPAN_SIZES = [128, 256]  # Uncomment to also test 256

# Random ablation settings
N_RANDOM_SPANS = 10  # Number of random spans per essay

# Top-k for overlap metric
TOP_K = 10

RANDOM_SEED = 42

print(f"Experiment: {EXPERIMENT}")
print(f"Cohort: {COHORT_PATH}")
print(f"\nContext: {CONTEXT_LENGTH} tokens")
print(f"Scoring regions: {SCORE_REGIONS}")
print(f"\nSpan locations: {list(SPAN_CONFIGS.keys())} + random")
print(f"Span sizes to test: {SPAN_SIZES}")
print(f"Random spans per essay: {N_RANDOM_SPANS}")
print(f"Top-k for overlap: {TOP_K}")

## 1. Load Model and Data

In [ ]:
# Load tokenizer
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Load model
print(f"Loading model: {MODEL_NAME}")

if USE_4BIT:
    print("  Using 4-bit quantization (T4 mode)")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto"
    )
else:
    print("  Using float16 (A100 mode)")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
    )

model.eval()
print("Model loaded")

In [ ]:
# Load cohort
def load_cohort(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

cohort = load_cohort(COHORT_PATH)
print(f"Loaded {len(cohort)} essays")

# Verify cohort
df_cohort = pd.DataFrame(cohort)
print(f"\nScore bin distribution:")
print(df_cohort['score_bin'].value_counts())

## 2. Core Functions

In [ ]:
@torch.no_grad()
def get_logits(token_ids):
    """
    Get logits for all positions in token_ids.
    """
    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    return outputs.logits[0]  # shape: (seq_len, vocab_size)


def compute_metrics_for_region(logits, token_ids, target_start, target_end):
    """
    Compute NLLs, top-1 predictions, and top-k sets for a scoring region.
    
    Args:
        logits: tensor of shape (seq_len, vocab_size)
        token_ids: list of token IDs
        target_start: start of scoring region (in token_ids indexing)
        target_end: end of scoring region
    
    Returns:
        dict with nlls, top1_preds, topk_sets
    """
    if target_end > len(token_ids):
        target_end = len(token_ids)
    if target_start >= target_end - 1:
        return {'nlls': [], 'top1_preds': [], 'topk_sets': []}
    
    nlls = []
    top1_preds = []
    topk_sets = []
    
    for i in range(target_start, target_end - 1):
        if i >= logits.shape[0]:
            break
        
        log_probs = torch.log_softmax(logits[i], dim=-1)
        true_token = token_ids[i + 1]
        nll = -log_probs[true_token].item()
        nlls.append(nll)
        
        # Top-1 prediction
        top1 = logits[i].argmax().item()
        top1_preds.append(top1)
        
        # Top-k set
        topk = set(logits[i].topk(TOP_K).indices.tolist())
        topk_sets.append(topk)
    
    return {
        'nlls': nlls,
        'top1_preds': top1_preds,
        'topk_sets': topk_sets,
    }


def compute_comparison_metrics(baseline, ablated, true_tokens):
    """
    Compare baseline vs ablated predictions.
    
    Returns:
        dict with delta_nll, flip_rate, topk_overlap_change
    """
    if not baseline['nlls'] or not ablated['nlls']:
        return None
    
    n = min(len(baseline['nlls']), len(ablated['nlls']))
    
    # Delta NLL (ablated - baseline, higher = ablation hurt more)
    delta_nll = np.mean(ablated['nlls'][:n]) - np.mean(baseline['nlls'][:n])
    
    # Flip rate (how often top-1 prediction changes)
    n_flips = sum(1 for i in range(n) if baseline['top1_preds'][i] != ablated['top1_preds'][i])
    flip_rate = n_flips / n
    
    # Top-k overlap change
    # Baseline: how much does top-k overlap with true token?
    # We measure: does ablation change the top-k set?
    topk_jaccard_changes = []
    for i in range(n):
        b_set = baseline['topk_sets'][i]
        a_set = ablated['topk_sets'][i]
        # Jaccard distance (0 = identical, 1 = disjoint)
        intersection = len(b_set & a_set)
        union = len(b_set | a_set)
        jaccard_sim = intersection / union if union > 0 else 1.0
        topk_jaccard_changes.append(1.0 - jaccard_sim)  # Convert to distance
    
    topk_change = np.mean(topk_jaccard_changes)
    
    return {
        'delta_nll': delta_nll,
        'flip_rate': flip_rate,
        'topk_change': topk_change,
        'n_tokens': n,
        'baseline_nll': np.mean(baseline['nlls'][:n]),
        'ablated_nll': np.mean(ablated['nlls'][:n]),
    }


def ablate_span_deletion(token_ids, span_start, span_end):
    """
    Delete span and concatenate remaining tokens.
    """
    return token_ids[:span_start] + token_ids[span_end:]


print("Core functions defined")

In [ ]:
def compute_span_boundaries_fixed_size(context_length, frac_start, span_size):
    """
    Compute span boundaries with fixed size starting at fractional position.
    
    Args:
        context_length: total context tokens (512)
        frac_start: starting position as fraction (0.0 to 1.0)
        span_size: fixed span size in tokens
    
    Returns:
        (start_idx, end_idx) clamped to valid range
    """
    span_start = int(context_length * frac_start)
    span_end = span_start + span_size
    
    # Clamp to context bounds
    if span_end > context_length:
        span_end = context_length
        span_start = max(0, span_end - span_size)
    
    return span_start, span_end


def run_span_ablation_for_essay(token_ids, essay_id, span_size, rng):
    """
    Run span ablation analysis for a single essay with stratified scoring.
    
    Args:
        token_ids: list of token IDs
        essay_id: for tracking
        span_size: size of ablation span in tokens
        rng: random.Random instance
    
    Returns:
        list of result dicts (one per span × score_region combination)
    """
    n_tokens = len(token_ids)
    
    # Need enough tokens for context + full scoring region
    min_required = CONTEXT_LENGTH + TOTAL_SCORE_TOKENS
    if n_tokens < min_required:
        return []
    
    # Full sequence for scoring: context + scoring region
    full_seq = token_ids[:min_required]
    
    # ============================================================
    # Baseline: full context
    # ============================================================
    baseline_logits = get_logits(full_seq)
    
    # Compute baseline metrics for each scoring region
    baseline_by_region = {}
    for region_name, (reg_start, reg_end) in SCORE_REGIONS.items():
        baseline_by_region[region_name] = compute_metrics_for_region(
            baseline_logits, full_seq, reg_start, reg_end
        )
    
    # True tokens for each region (for reference)
    true_tokens_by_region = {}
    for region_name, (reg_start, reg_end) in SCORE_REGIONS.items():
        true_tokens_by_region[region_name] = full_seq[reg_start+1:reg_end]
    
    results = []
    
    # ============================================================
    # Fixed-position spans
    # ============================================================
    for span_label, (frac_start, frac_end) in SPAN_CONFIGS.items():
        span_start, span_end = compute_span_boundaries_fixed_size(
            CONTEXT_LENGTH, frac_start, span_size
        )
        actual_span_size = span_end - span_start
        
        # Ablate context
        ablated_context = ablate_span_deletion(full_seq[:CONTEXT_LENGTH], span_start, span_end)
        
        # Reconstruct: ablated context + original scoring tokens
        ablated_full = ablated_context + full_seq[CONTEXT_LENGTH:]
        
        # Get ablated logits
        ablated_logits = get_logits(ablated_full)
        
        # Compute metrics for each scoring region
        # Note: positions shifted by (span_size) tokens
        shift = CONTEXT_LENGTH - len(ablated_context)
        
        for region_name, (reg_start, reg_end) in SCORE_REGIONS.items():
            # Adjusted positions in ablated sequence
            adj_start = reg_start - shift
            adj_end = reg_end - shift
            
            ablated_metrics = compute_metrics_for_region(
                ablated_logits, ablated_full, adj_start, adj_end
            )
            
            comparison = compute_comparison_metrics(
                baseline_by_region[region_name],
                ablated_metrics,
                true_tokens_by_region[region_name]
            )
            
            if comparison:
                results.append({
                    'essay_id': essay_id,
                    'span_label': span_label,
                    'span_size': actual_span_size,
                    'span_start_token': span_start,
                    'span_end_token': span_end,
                    'score_region': region_name,
                    'delta_nll': comparison['delta_nll'],
                    'flip_rate': comparison['flip_rate'],
                    'topk_change': comparison['topk_change'],
                    'baseline_nll': comparison['baseline_nll'],
                    'ablated_nll': comparison['ablated_nll'],
                    'n_scored_tokens': comparison['n_tokens'],
                })
    
    # ============================================================
    # Random spans (N_RANDOM_SPANS per essay, averaged later)
    # ============================================================
    random_results_by_region = defaultdict(list)
    
    for rand_idx in range(N_RANDOM_SPANS):
        # Random start position
        max_start = CONTEXT_LENGTH - span_size
        if max_start <= 0:
            continue
        span_start = rng.randint(0, max_start)
        span_end = span_start + span_size
        
        # Ablate
        ablated_context = ablate_span_deletion(full_seq[:CONTEXT_LENGTH], span_start, span_end)
        ablated_full = ablated_context + full_seq[CONTEXT_LENGTH:]
        ablated_logits = get_logits(ablated_full)
        
        shift = CONTEXT_LENGTH - len(ablated_context)
        
        for region_name, (reg_start, reg_end) in SCORE_REGIONS.items():
            adj_start = reg_start - shift
            adj_end = reg_end - shift
            
            ablated_metrics = compute_metrics_for_region(
                ablated_logits, ablated_full, adj_start, adj_end
            )
            
            comparison = compute_comparison_metrics(
                baseline_by_region[region_name],
                ablated_metrics,
                true_tokens_by_region[region_name]
            )
            
            if comparison:
                random_results_by_region[region_name].append(comparison)
    
    # Average random results within essay
    for region_name, comparisons in random_results_by_region.items():
        if comparisons:
            results.append({
                'essay_id': essay_id,
                'span_label': 'random',
                'span_size': span_size,
                'span_start_token': np.nan,  # Averaged across positions
                'span_end_token': np.nan,
                'score_region': region_name,
                'delta_nll': np.mean([c['delta_nll'] for c in comparisons]),
                'flip_rate': np.mean([c['flip_rate'] for c in comparisons]),
                'topk_change': np.mean([c['topk_change'] for c in comparisons]),
                'baseline_nll': np.mean([c['baseline_nll'] for c in comparisons]),
                'ablated_nll': np.mean([c['ablated_nll'] for c in comparisons]),
                'n_scored_tokens': comparisons[0]['n_tokens'],
                'n_random_samples': len(comparisons),
            })
    
    return results


print("Span ablation function defined")

## 3. Run Span Ablation

In [ ]:
# Pre-tokenize all essays
essay_tokens = {}
for essay in cohort:
    token_ids = tokenizer.encode(essay['text'], add_special_tokens=False)
    essay_tokens[essay['essay_id']] = token_ids

print(f"Tokenized {len(essay_tokens)} essays")

# Token length stats
lengths = [len(t) for t in essay_tokens.values()]
print(f"Token lengths: min={min(lengths)}, median={np.median(lengths):.0f}, max={max(lengths)}")

In [ ]:
# Process all essays
all_results = []
start_time = time.time()

for span_size in SPAN_SIZES:
    print(f"\n{'='*60}")
    print(f"Running with span_size = {span_size} tokens")
    print(f"{'='*60}")
    
    rng = random.Random(RANDOM_SEED)
    
    n_spans = len(SPAN_CONFIGS) + 1  # +1 for random
    n_regions = len(SCORE_REGIONS)
    expected_rows = len(cohort) * n_spans * n_regions
    print(f"Expected: {len(cohort)} essays × {n_spans} spans × {n_regions} regions = {expected_rows} rows")
    print(f"Plus {N_RANDOM_SPANS} random samples per essay (averaged)")
    
    for essay in tqdm(cohort, desc=f"Span size {span_size}"):
        essay_id = essay['essay_id']
        token_ids = essay_tokens[essay_id]
        
        # Run span ablation
        results = run_span_ablation_for_essay(token_ids, essay_id, span_size, rng)
        
        # Add essay metadata
        for r in results:
            r['score'] = essay.get('score')
            r['score_bin'] = essay.get('score_bin')
            r['grade'] = essay.get('grade')
            r['token_count'] = len(token_ids)
        
        all_results.extend(results)

elapsed = time.time() - start_time
print(f"\nDone in {elapsed:.1f}s ({elapsed/len(cohort):.2f}s/essay)")
print(f"Total result rows: {len(all_results)}")

In [ ]:
# Create DataFrame
df = pd.DataFrame(all_results)

print(f"Results shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nSpan label × Score region counts:")
print(pd.crosstab(df['span_label'], df['score_region']))

print(f"\nSample rows:")
df.head(10)

## 4. Group Summaries

In [ ]:
# Define ordering
GROUP_ORDER = ['low', 'mid', 'high']
SPAN_ORDER = ['early', 'early_mid', 'middle', 'late', 'pre_target', 'immediate_pre_target', 'random']
REGION_ORDER = ['score_early', 'score_mid', 'score_late']

# Set categorical ordering
df['score_bin'] = pd.Categorical(df['score_bin'], categories=GROUP_ORDER, ordered=True)
df['span_label'] = pd.Categorical(df['span_label'], categories=SPAN_ORDER, ordered=True)
df['score_region'] = pd.Categorical(df['score_region'], categories=REGION_ORDER, ordered=True)

In [ ]:
# Summary by score_bin × span_label × score_region
print("="*80)
print("FLIP RATE BY SCORE_BIN × SPAN_LABEL × SCORE_REGION (PRIMARY METRIC)")
print("="*80)
print("\nHigher flip_rate = ablating that span changes more word predictions")

summary_rows = []

for score_bin in GROUP_ORDER:
    for span_label in SPAN_ORDER:
        for region in REGION_ORDER:
            subset = df[(df['score_bin'] == score_bin) & 
                       (df['span_label'] == span_label) & 
                       (df['score_region'] == region)]
            if len(subset) == 0:
                continue
            
            summary_rows.append({
                'score_bin': score_bin,
                'span_label': span_label,
                'score_region': region,
                'n': len(subset),
                'flip_rate_mean': subset['flip_rate'].mean(),
                'flip_rate_sem': subset['flip_rate'].sem(),
                'delta_nll_mean': subset['delta_nll'].mean(),
                'delta_nll_sem': subset['delta_nll'].sem(),
                'topk_change_mean': subset['topk_change'].mean(),
                'topk_change_sem': subset['topk_change'].sem(),
            })

df_summary = pd.DataFrame(summary_rows)
print(f"\nSummary table: {len(df_summary)} rows")
df_summary.head(20)

In [ ]:
# Pivot: flip_rate by span_label (rows) × score_region (columns), faceted by score_bin
print("\n" + "="*80)
print("PIVOT: Mean Flip Rate")
print("="*80)

for score_bin in GROUP_ORDER:
    print(f"\n--- Score bin: {score_bin} ---")
    subset = df[df['score_bin'] == score_bin]
    pivot = subset.pivot_table(
        values='flip_rate',
        index='span_label',
        columns='score_region',
        aggfunc='mean'
    )
    if REGION_ORDER[0] in pivot.columns:
        pivot = pivot[REGION_ORDER]
    print(pivot.round(4))

In [ ]:
# Key comparison: boundary-adjacent spans vs others
print("\n" + "="*80)
print("KEY TEST: Boundary-Adjacent Spans on score_early")
print("="*80)
print("\nHypothesis: immediate_pre_target should have highest impact on score_early")

early_region = df[df['score_region'] == 'score_early']

print(f"\n{'Span Label':<25} {'Mean Flip Rate':>15} {'Mean ΔNLL':>12} {'Mean Top-k Δ':>12}")
print("-"*70)

for span_label in SPAN_ORDER:
    subset = early_region[early_region['span_label'] == span_label]
    if len(subset) > 0:
        print(f"{span_label:<25} {subset['flip_rate'].mean():>15.4f} "
              f"{subset['delta_nll'].mean():>12.4f} {subset['topk_change'].mean():>12.4f}")

## 5. Statistical Tests

In [ ]:
# Standardize token count
df['token_count_z'] = (df['token_count'] - df['token_count'].mean()) / df['token_count'].std()

print("="*80)
print("REGRESSION: flip_rate ~ span_label + score_region + score_bin + interactions")
print("="*80)

# Full model
formula = ('flip_rate ~ C(span_label) + C(score_region) + C(score_bin) + '
           'C(span_label):C(score_region) + C(span_label):C(score_bin) + token_count_z')

model_full = smf.ols(formula, data=df).fit()

print(f"\nFormula: {formula}")
print(f"R²: {model_full.rsquared:.4f}, Adj R²: {model_full.rsquared_adj:.4f}, n={int(model_full.nobs)}")
print(f"\nCoefficients (selected):")

# Show key coefficients
key_params = [p for p in model_full.params.index if 
              'immediate_pre_target' in p or 'pre_target' in p or 'score_early' in p]
for param in key_params[:10]:
    coef = model_full.params[param]
    pval = model_full.pvalues[param]
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
    print(f"  {param}: β={coef:+.4f}, p={pval:.4f} {sig}")

In [ ]:
# Test: Does span×region interaction exist?
print("\n" + "="*80)
print("MODEL COMPARISON: With vs Without Span×Region Interaction")
print("="*80)

formula_no_interaction = ('flip_rate ~ C(span_label) + C(score_region) + C(score_bin) + '
                          'C(span_label):C(score_bin) + token_count_z')
model_reduced = smf.ols(formula_no_interaction, data=df).fit()

print(f"\nFull model R²: {model_full.rsquared:.4f}")
print(f"Reduced model R²: {model_reduced.rsquared:.4f}")
print(f"R² difference: {model_full.rsquared - model_reduced.rsquared:.4f}")

# F-test for nested models
from scipy import stats as scipy_stats

ssr_full = model_full.ssr
ssr_reduced = model_reduced.ssr
df_full = model_full.df_resid
df_diff = model_reduced.df_resid - model_full.df_resid

f_stat = ((ssr_reduced - ssr_full) / df_diff) / (ssr_full / df_full)
p_value = 1 - scipy_stats.f.cdf(f_stat, df_diff, df_full)

print(f"\nF-test for span×region interaction:")
print(f"  F({df_diff}, {df_full}) = {f_stat:.2f}, p = {p_value:.4e}")
print(f"  {'Significant' if p_value < 0.05 else 'Not significant'}: span effect varies by scoring region")

In [ ]:
# Focused test: immediate_pre_target effect on score_early vs score_late
print("\n" + "="*80)
print("FOCUSED TEST: Boundary-Adjacent Span Impact Decay")
print("="*80)

# Compare immediate_pre_target effect across scoring regions
imm_pre = df[df['span_label'] == 'immediate_pre_target']

print("\nimmediate_pre_target ablation effect by scoring region:")
print(f"{'Region':<15} {'Mean Flip Rate':>15} {'95% CI':>20}")
print("-"*55)

for region in REGION_ORDER:
    subset = imm_pre[imm_pre['score_region'] == region]
    if len(subset) > 0:
        mean = subset['flip_rate'].mean()
        sem = subset['flip_rate'].sem()
        ci = 1.96 * sem
        print(f"{region:<15} {mean:>15.4f} [{mean-ci:.4f}, {mean+ci:.4f}]")

# T-test: score_early vs score_late
early = imm_pre[imm_pre['score_region'] == 'score_early']['flip_rate']
late = imm_pre[imm_pre['score_region'] == 'score_late']['flip_rate']

t_stat, p_val = scipy_stats.ttest_ind(early, late)
print(f"\nT-test (score_early vs score_late): t={t_stat:.2f}, p={p_val:.4f}")
print(f"Effect decays with distance: {'Yes' if early.mean() > late.mean() and p_val < 0.05 else 'Inconclusive'}")

## 6. Visualization

In [ ]:
# Color schemes
SCORE_COLORS = {'low': '#e74c3c', 'mid': '#f39c12', 'high': '#2ecc71'}
REGION_COLORS = {'score_early': '#3498db', 'score_mid': '#9b59b6', 'score_late': '#1abc9c'}

In [ ]:
# Main plot: Flip rate by span location, faceted by score region
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

x_positions = np.arange(len(SPAN_ORDER))
width = 0.25

for ax_idx, region in enumerate(REGION_ORDER):
    ax = axes[ax_idx]
    region_data = df[df['score_region'] == region]
    
    for i, score_bin in enumerate(GROUP_ORDER):
        means = []
        cis = []
        for span in SPAN_ORDER:
            subset = region_data[(region_data['score_bin'] == score_bin) & 
                                (region_data['span_label'] == span)]
            if len(subset) > 0:
                means.append(subset['flip_rate'].mean())
                cis.append(1.96 * subset['flip_rate'].sem())
            else:
                means.append(np.nan)
                cis.append(np.nan)
        
        offset = (i - 1) * width
        ax.bar(x_positions + offset, means, width, yerr=cis,
               label=score_bin, color=SCORE_COLORS[score_bin], capsize=2, alpha=0.8)
    
    ax.set_title(f'{region}\n(tokens {SCORE_REGIONS[region][0]}-{SCORE_REGIONS[region][1]})', 
                 fontsize=11, fontweight='bold')
    ax.set_xticks(x_positions)
    ax.set_xticklabels(['early', 'early\nmid', 'middle', 'late', 'pre\ntarget', 'immed\npre', 'random'],
                       fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')
    
    if ax_idx == 0:
        ax.set_ylabel('Flip Rate', fontsize=11)
    if ax_idx == 1:
        ax.legend(title='Score Bin', loc='upper left', fontsize=9)

fig.suptitle('Span Ablation: Flip Rate by Context Location and Scoring Region\n'
             '(Higher = ablating that span changes more word predictions)', 
             fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig(Path(OUTPUT_BASE) / EXPERIMENT / 'flip_rate_by_span_region.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap: Mean flip rate (span × region), averaged across score bins
fig, ax = plt.subplots(figsize=(10, 6))

pivot_flip = df.pivot_table(
    values='flip_rate',
    index='span_label',
    columns='score_region',
    aggfunc='mean'
)[REGION_ORDER]

sns.heatmap(pivot_flip, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax,
            cbar_kws={'label': 'Flip Rate'})
ax.set_title('Mean Flip Rate: Ablated Span Location × Scoring Region\n'
             '(Hypothesis: bottom-right should be highest)', fontsize=12, fontweight='bold')
ax.set_xlabel('Scoring Region (distance from context)', fontsize=11)
ax.set_ylabel('Ablated Span Location', fontsize=11)

plt.tight_layout()
plt.savefig(Path(OUTPUT_BASE) / EXPERIMENT / 'flip_rate_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Line plot: Decay of boundary-adjacent span effect
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: immediate_pre_target effect by region
ax = axes[0]
imm_pre = df[df['span_label'] == 'immediate_pre_target']

for score_bin in GROUP_ORDER:
    means = []
    cis = []
    for region in REGION_ORDER:
        subset = imm_pre[(imm_pre['score_bin'] == score_bin) & (imm_pre['score_region'] == region)]
        if len(subset) > 0:
            means.append(subset['flip_rate'].mean())
            cis.append(1.96 * subset['flip_rate'].sem())
        else:
            means.append(np.nan)
            cis.append(np.nan)
    
    ax.errorbar(range(len(REGION_ORDER)), means, yerr=cis, marker='o', capsize=4,
                label=score_bin, color=SCORE_COLORS[score_bin], linewidth=2, markersize=8)

ax.set_xticks(range(len(REGION_ORDER)))
ax.set_xticklabels(['Early\n(512-576)', 'Mid\n(576-640)', 'Late\n(640-768)'])
ax.set_xlabel('Scoring Region (distance from ablated span)', fontsize=11)
ax.set_ylabel('Flip Rate', fontsize=11)
ax.set_title('Immediate Pre-Target Ablation:\nEffect Decay with Distance', fontsize=12, fontweight='bold')
ax.legend(title='Score Bin')
ax.grid(True, alpha=0.3)

# Plot 2: Compare boundary-adjacent vs early span
ax = axes[1]
early_span = df[df['span_label'] == 'early']

# Just show score_early region
score_early_data = df[df['score_region'] == 'score_early']

for score_bin in GROUP_ORDER:
    means = []
    for span in ['early', 'early_mid', 'middle', 'late', 'pre_target', 'immediate_pre_target']:
        subset = score_early_data[(score_early_data['score_bin'] == score_bin) & 
                                  (score_early_data['span_label'] == span)]
        if len(subset) > 0:
            means.append(subset['flip_rate'].mean())
        else:
            means.append(np.nan)
    
    ax.plot(range(6), means, marker='s', label=score_bin, 
            color=SCORE_COLORS[score_bin], linewidth=2, markersize=8)

ax.set_xticks(range(6))
ax.set_xticklabels(['early', 'early_mid', 'middle', 'late', 'pre_target', 'immed_pre'], rotation=45)
ax.set_xlabel('Ablated Span Location (left=distant, right=adjacent)', fontsize=11)
ax.set_ylabel('Flip Rate on score_early', fontsize=11)
ax.set_title('score_early Region:\nSpan Distance vs Impact', fontsize=12, fontweight='bold')
ax.legend(title='Score Bin')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(Path(OUTPUT_BASE) / EXPERIMENT / 'effect_decay_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top-k change visualization
fig, ax = plt.subplots(figsize=(12, 6))

# Focus on score_early region
score_early = df[df['score_region'] == 'score_early']

x_positions = np.arange(len(SPAN_ORDER))
width = 0.25

for i, score_bin in enumerate(GROUP_ORDER):
    means = []
    cis = []
    for span in SPAN_ORDER:
        subset = score_early[(score_early['score_bin'] == score_bin) & 
                            (score_early['span_label'] == span)]
        if len(subset) > 0:
            means.append(subset['topk_change'].mean())
            cis.append(1.96 * subset['topk_change'].sem())
        else:
            means.append(np.nan)
            cis.append(np.nan)
    
    offset = (i - 1) * width
    ax.bar(x_positions + offset, means, width, yerr=cis,
           label=score_bin, color=SCORE_COLORS[score_bin], capsize=3, alpha=0.8)

ax.set_xlabel('Ablated Span Location', fontsize=12)
ax.set_ylabel(f'Top-{TOP_K} Set Change (Jaccard Distance)', fontsize=12)
ax.set_title(f'Top-{TOP_K} Prediction Set Change by Span Location\n'
             f'(score_early region only)', fontsize=13, fontweight='bold')
ax.set_xticks(x_positions)
ax.set_xticklabels(['early', 'early_mid', 'middle', 'late', 'pre_target', 'immed_pre', 'random'])
ax.legend(title='Score Bin')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(Path(OUTPUT_BASE) / EXPERIMENT / 'topk_change_by_span.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Save Results

In [ ]:
# Create output directory
output_dir = Path(OUTPUT_BASE) / EXPERIMENT
output_dir.mkdir(parents=True, exist_ok=True)

# Save full results
df.to_csv(output_dir / 'span_ablation_v2_results.csv', index=False)

# Save summary
df_summary.to_csv(output_dir / 'span_ablation_v2_summary.csv', index=False)

# Save pivot tables
pivot_flip.to_csv(output_dir / 'pivot_flip_rate.csv')

# Save regression results
with open(output_dir / 'regression_v2.txt', 'w') as f:
    f.write("ENHANCED SPAN ABLATION REGRESSION ANALYSIS (v2)\n")
    f.write("="*80 + "\n\n")
    f.write(f"Configuration:\n")
    f.write(f"  Span sizes: {SPAN_SIZES}\n")
    f.write(f"  Span locations: {list(SPAN_CONFIGS.keys())} + random\n")
    f.write(f"  Scoring regions: {SCORE_REGIONS}\n")
    f.write(f"  Random samples per essay: {N_RANDOM_SPANS}\n")
    f.write(f"  Top-k: {TOP_K}\n")
    f.write("\n" + "="*80 + "\n\n")
    f.write("Full Model:\n")
    f.write(f"Formula: {formula}\n")
    f.write(model_full.summary().as_text())

print(f"\nSaved to {output_dir}/")
print(f"  - span_ablation_v2_results.csv ({len(df)} rows)")
print(f"  - span_ablation_v2_summary.csv")
print(f"  - pivot_flip_rate.csv")
print(f"  - regression_v2.txt")
print(f"  - flip_rate_by_span_region.png")
print(f"  - flip_rate_heatmap.png")
print(f"  - effect_decay_analysis.png")
print(f"  - topk_change_by_span.png")

In [ ]:
# Final summary
print("\n" + "="*80)
print("SUMMARY OF FINDINGS")
print("="*80)

print("""
KEY QUESTIONS ADDRESSED:

1. Does boundary-adjacent context matter most for nearby predictions?
   → Compare immediate_pre_target effect on score_early vs score_late
   → Expected: Higher flip_rate on score_early, decay with distance

2. Do higher-quality essays show different context usage patterns?
   → Compare score_bin effects across span×region combinations
   → If high-quality essays use distant context more, they should show:
     - Relatively higher early/middle span effects
     - Less steep decay with distance

3. Is flip_rate a better metric than delta_nll for "specific word choice"?
   → flip_rate directly measures prediction changes
   → topk_change captures subtler distributional shifts

INTERPRETATION GUIDE:
- High flip_rate = ablating that context changes many word predictions
- Decay pattern = how context influence diminishes with distance
- Score×span interaction = different writing quality uses context differently
""")